In [ ]:
# ============================================================
# TRAIN-TEST LEAKAGE CHECK
# Exact duplicate + near-duplicate (perceptual hash) detection
# ============================================================
!pip install imagehash --quiet

import os
import hashlib
import imagehash
from PIL import Image
import pandas as pd
import numpy as np
from collections import defaultdict

OUTPUT_DIR = "/kaggle/working"          # <-- FIX: /kaggle/input/... always read-only
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ---------- 1. EXACT PATH / FILENAME OVERLAP ----------
train_set = set(train_paths)
val_set = set(val_paths)

exact_path_overlap = train_set & val_set
print(f"Exact path overlap (same file in both splits): {len(exact_path_overlap)}")

train_fnames = set(os.path.basename(p) for p in train_paths)
val_fnames = set(os.path.basename(p) for p in val_paths)
fname_overlap = train_fnames & val_fnames
print(f"Filename overlap (possible copy-with-different-folder): {len(fname_overlap)}")

# ---------- 2. EXACT BYTE-LEVEL DUPLICATE CHECK (MD5) ----------
def md5_of_file(path, chunk_size=8192):
    h = hashlib.md5()
    try:
        with open(path, "rb") as f:
            for chunk in iter(lambda: f.read(chunk_size), b""):
                h.update(chunk)
        return h.hexdigest()
    except Exception:
        return None

print("\nComputing MD5 hashes for exact byte-duplicate check...")
train_md5 = {p: md5_of_file(p) for p in train_paths}
val_md5 = {p: md5_of_file(p) for p in val_paths}

train_md5_to_path = defaultdict(list)
for p, h in train_md5.items():
    if h:
        train_md5_to_path[h].append(p)

exact_byte_dupes = []
for vp, vh in val_md5.items():
    if vh in train_md5_to_path:
        for tp in train_md5_to_path[vh]:
            exact_byte_dupes.append((tp, vp))

print(f"Exact byte-identical images across train/val: {len(exact_byte_dupes)}")

# ---------- 3. NEAR-DUPLICATE CHECK (PERCEPTUAL HASH) ----------
def phash_of(path, hash_size=16):
    try:
        img = Image.open(path).convert("RGB")
        return imagehash.phash(img, hash_size=hash_size)
    except Exception:
        return None

print("\nComputing perceptual hashes (this can take a while for large datasets)...")
train_phash = {p: phash_of(p) for p in train_paths}
val_phash = {p: phash_of(p) for p in val_paths}

train_phash = {p: h for p, h in train_phash.items() if h is not None}
val_phash = {p: h for p, h in val_phash.items() if h is not None}

HAMMING_THRESHOLD = 5  # <=5 bits diff = visually near-identical

near_dupes = []
train_items = list(train_phash.items())
for vp, vh in val_phash.items():
    for tp, th in train_items:
        if (vh - th) <= HAMMING_THRESHOLD:
            near_dupes.append((tp, vp, vh - th))

near_dupes_df = pd.DataFrame(near_dupes, columns=["train_path", "val_path", "hamming_distance"])
near_dupes_df.to_csv(f"{OUTPUT_DIR}/leakage_near_duplicates.csv", index=False)

print(f"Near-duplicate image pairs (hamming <= {HAMMING_THRESHOLD}): {len(near_dupes)}")
if len(near_dupes) > 0:
    affected_val = near_dupes_df["val_path"].nunique()
    print(f"Affected val images: {affected_val} / {len(val_paths)} "
          f"({100*affected_val/len(val_paths):.2f}%)")

# ---------- 4. CLASS-LEVEL LEAKAGE SUMMARY ----------
summary = {
    "n_train": len(train_paths),
    "n_val": len(val_paths),
    "exact_path_overlap": len(exact_path_overlap),
    "filename_overlap": len(fname_overlap),
    "exact_byte_duplicates": len(exact_byte_dupes),
    "near_duplicate_pairs": len(near_dupes),
    "near_duplicate_affected_val_pct": round(
        100 * near_dupes_df["val_path"].nunique() / len(val_paths), 2
    ) if len(near_dupes) > 0 else 0.0,
}
pd.DataFrame([summary]).to_csv(f"{OUTPUT_DIR}/leakage_summary.csv", index=False)

print("\n======================")
print("LEAKAGE CHECK SUMMARY")
print("======================")
for k, v in summary.items():
    print(f"{k}: {v}")

if summary["exact_byte_duplicates"] == 0 and summary["near_duplicate_pairs"] == 0:
    print("\n✅ No train/val leakage detected.")
else:
    print("\n⚠️ Potential leakage found — review leakage_near_duplicates.csv before reporting results.")